# Занятие 2. Изображение как массив и тензор
## Размерности, каналы и форматы данных

**Часть 1** — демонстрация преподавателя.  
**Часть 2** — самостоятельные операции по 1–2 строки в `# YOUR CODE HERE`.

## Часть 1. Демонстрация преподавателя

Цель — буквально увидеть, что после загрузки фотография становится массивом чисел, и научиться читать его форму.

In [ ]:
import os
import tempfile
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.datasets import load_sample_image

plt.rcParams['figure.figsize'] = (8, 4)

### 1. Загружаем готовое RGB-изображение
`load_sample_image` берёт пример, который уже входит в scikit-learn — интернет не нужен.

In [ ]:
image = load_sample_image('china.jpg')

print('type :', type(image))
print('shape:', image.shape)
print('dtype:', image.dtype)
print('min/max:', image.min(), image.max())

plt.imshow(image)
plt.axis('off')
plt.show()

### 2. Что означает shape `(H, W, C)`?
- `H` — высота в пикселях;
- `W` — ширина;
- `C` — число каналов. Для RGB их три.

In [ ]:
H, W, C = image.shape
print('Высота:', H)
print('Ширина:', W)
print('Каналы:', C)
print('Пикселей:', H * W)
print('Числовых значений:', H * W * C)
print('Память массива, байт:', image.nbytes)

### 3. Один пиксель — это три числа
Посмотрим на пиксель с координатами `(y=100, x=200)`.

In [ ]:
y, x = 100, 200
pixel = image[y, x]
print('RGB:', pixel)

patch = np.zeros((40, 120, 3), dtype=np.uint8)
patch[:] = pixel
plt.imshow(patch)
plt.title(f'RGB = {pixel.tolist()}')
plt.axis('off')
plt.show()

### 4. Каналы можно рассматривать отдельно
Три канала имеют одинаковую высоту и ширину, но хранят разную компоненту цвета.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, channel, name in zip(axes, range(3), ['R', 'G', 'B']):
    ax.imshow(image[:, :, channel], cmap='gray')
    ax.set_title(f'Канал {name}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### 5. `uint8`: почему значения от 0 до 255?
Один канал хранит 256 возможных значений. У RGB-пикселя три канала, поэтому при 8 битах на канал возможно `256³ = 16 777 216` комбинаций цветов.

In [ ]:
print('Возможных значений одного uint8-канала:', 256)
print('Возможных RGB-комбинаций:', 256 ** 3)

### 6. Нормализация меняет числа, но не геометрию
Переведём значения 0..255 в диапазон 0..1.

In [ ]:
image_float = image.astype(np.float32) / 255.0
print('shape до/после:', image.shape, image_float.shape)
print('dtype до/после:', image.dtype, image_float.dtype)
print('диапазон:', image_float.min(), image_float.max())

### 7. Один и тот же массив — разные порядки осей
Многие Python-библиотеки используют `(H, W, C)`, а нейросетевые фреймворки часто работают с `(C, H, W)`. Это **перестановка осей**, а не изменение картинки.

In [ ]:
image_chw = image.transpose(2, 0, 1)
print('HWC:', image.shape)
print('CHW:', image_chw.shape)

batch = image_chw[None, ...]
print('NCHW для одного изображения:', batch.shape)

### 8. Файл изображения и массив — не одно и то же
PNG и JPEG — способы хранения файла. После декодирования мы снова получаем массив. JPEG использует сжатие с потерями, поэтому отдельные пиксели могут немного измениться.

In [ ]:
crop = image[80:280, 180:480]

with tempfile.TemporaryDirectory() as tmp:
    png_path = os.path.join(tmp, 'sample.png')
    jpg_path = os.path.join(tmp, 'sample.jpg')
    Image.fromarray(crop).save(png_path)
    Image.fromarray(crop).save(jpg_path, quality=80)

    png = np.array(Image.open(png_path).convert('RGB'))
    jpg = np.array(Image.open(jpg_path).convert('RGB'))

    print('Размер PNG, байт :', os.path.getsize(png_path))
    print('Размер JPEG, байт:', os.path.getsize(jpg_path))
    print('Форма после загрузки:', png.shape, jpg.shape)
    print('Средняя |разница пикселей| PNG vs исходник :', np.abs(png.astype(int)-crop.astype(int)).mean())
    print('Средняя |разница пикселей| JPEG vs исходник:', np.abs(jpg.astype(int)-crop.astype(int)).mean())

---
## Часть 2. Самостоятельные мини-задачи

Работаем с уже загруженным `image`. Каждый шаг — одна или две строки.

### Задание 1. Прочитайте shape
Запишите высоту, ширину и число каналов в переменные `h`, `w`, `c`.

In [ ]:
# YOUR CODE HERE (1 строка)
# h, w, c = ...

### Задание 2. Сколько пикселей?
Посчитайте число пикселей в изображении. Каналы здесь **не** умножаем.

In [ ]:
# YOUR CODE HERE (1 строка)
# n_pixels = ...

### Задание 3. Сколько числовых значений?
Теперь посчитайте количество всех компонент RGB.

In [ ]:
# YOUR CODE HERE (1 строка)
# n_values = ...

### Задание 4. Достаньте зелёный канал
Создайте двумерный массив `green` только со значениями G-канала.

In [ ]:
# YOUR CODE HERE (1 строка)
# green = ...

### Задание 5. Посмотрите на один пиксель
Получите RGB-вектор пикселя в строке 50 и столбце 100.

In [ ]:
# YOUR CODE HERE (1 строка)
# pixel_50_100 = ...

### Задание 6. Нормализуйте изображение
Сделайте `float32`-копию со значениями 0..1.

In [ ]:
# YOUR CODE HERE (1 строка)
# normalized = ...

### Задание 7. Переставьте оси HWC → CHW
Создайте `chw`.

In [ ]:
# YOUR CODE HERE (1 строка)
# chw = ...

### Задание 8. Добавьте размерность batch
Из массива `(C, H, W)` сделайте `(1, C, H, W)`.

In [ ]:
# YOUR CODE HERE (1 строка)
# batch1 = ...

### Задание 9. Средний цвет изображения
Получите три средних значения каналов R, G, B.

In [ ]:
# YOUR CODE HERE (1 строка)
# mean_rgb = ...

### Мини-вопрос
Что изменится, если просто выполнить `reshape(3, H, W)` вместо `transpose(2, 0, 1)`? Почему одинаковая итоговая форма ещё не означает, что данные расположены правильно?